# Neural Network Architectures in TensorFlow/Keras

TensorFlow/Keras companion to the PyTorch lab. Re-implements each architecture as custom Keras layers and models, with emphasis on the Keras subclassing API, `training=` flag propagation, and `GradientTape` for gradient inspection.

| Concept | PyTorch | TensorFlow / Keras |
|---------|---------|--------------------|
| Model definition | `nn.Module` + `forward` | `tf.keras.layers.Layer` + `call` |
| Sequential model | `nn.Sequential` | `tf.keras.Sequential` |
| Gradient computation | `loss.backward()` | `tf.GradientTape` |
| Gradient access | `param.grad` | `tape.gradient(loss, vars)` |
| Data channels order | NCHW (default) | NHWC (default) |
| Causal mask | Upper-triangular bool | Additive −1e9 mask |

| Section | Topic | Key experiment |
|---------|-------|----------------|
| 1 | MLP with Keras subclassing | Rank collapse without activations |
| 2 | CNN and depthwise separable | Parameter counts; CIFAR-10 accuracy |
| 3 | ResNet with GradientTape | GradientTape gradient-flow visualisation |
| 4 | Vanilla RNN vs. LSTM | Long-sequence task; gradient norms |
| 5 | Scaled dot-product attention | Saturation experiment; MHA module |
| 6 | Mini decoder-only transformer | Character LM; per-head attention heatmaps |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42)
print(f'TensorFlow {tf.__version__}')

## Section 1 — MLP with Keras Subclassing

We build a configurable MLP as a `tf.keras.layers.Layer` and demonstrate the **rank collapse** property of linear (activation-free) networks: a stack of linear layers is equivalent to a single matrix multiplication, so output rank ≤ `d_in` regardless of hidden width.

In [ ]:
class MLP(tf.keras.layers.Layer):
    def __init__(self, hidden_dims, d_out, activation='relu'):
        super().__init__()
        self.hidden = [tf.keras.layers.Dense(d, activation=activation)
                       for d in hidden_dims]
        self.out    = tf.keras.layers.Dense(d_out, activation=None)

    def call(self, x):
        for layer in self.hidden:
            x = layer(x)
        return self.out(x)


def count_params_layer(layer, x_shape):
    """Build the layer by calling it once, then count parameters."""
    layer(tf.zeros(x_shape))
    return layer.count_params()


def expected_params(d_in, hidden_dims, d_out):
    dims = [d_in] + list(hidden_dims) + [d_out]
    return sum(dims[i] * dims[i+1] + dims[i+1] for i in range(len(dims) - 1))


print('Parameter count verification:')
for d_in, hidden, d_out in [(16, [64, 64], 10), (784, [256, 128, 64], 10)]:
    m = MLP(hidden, d_out)
    actual   = count_params_layer(m, (1, d_in))
    expected = expected_params(d_in, hidden, d_out)
    ok = '✓' if actual == expected else '✗'
    print(f'  {ok} MLP({d_in}, {hidden}, {d_out}): actual={actual}  expected={expected}')

In [ ]:
# Rank collapse experiment
tf.random.set_seed(0)
d_in, d_out = 8, 32
hidden = [64, 64, 64]

mlp_linear = MLP(hidden, d_out, activation=None)
mlp_relu   = MLP(hidden, d_out, activation='relu')

x = tf.random.normal((256, d_in))
out_linear = mlp_linear(x, training=False)
out_relu   = mlp_relu(x,   training=False)

rank_linear = tf.linalg.matrix_rank(out_linear).numpy()
rank_relu   = tf.linalg.matrix_rank(out_relu).numpy()

print(f'Linear MLP (no activations): output rank = {rank_linear}  (max possible = {d_in})')
print(f'ReLU MLP:                    output rank = {rank_relu}')
assert rank_linear <= d_in, 'Expected rank ≤ d_in for linear MLP'
print(f'\n✓ Linear MLP rank ({rank_linear}) ≤ d_in ({d_in}) — collapse confirmed')

## Section 2 — CNN and Depthwise Separable Convolutions

TF/Keras uses **NHWC** layout by default (batch, height, width, channels). `tf.keras.layers.DepthwiseConv2D` implements the per-channel depthwise step directly.

In [ ]:
class ConvBlock(tf.keras.layers.Layer):
    def __init__(self, out_ch, K=3, strides=1):
        super().__init__()
        self.conv = tf.keras.layers.Conv2D(out_ch, K, strides=strides, padding='same', use_bias=False)
        self.bn   = tf.keras.layers.BatchNormalization()

    def call(self, x, training=False):
        return tf.nn.relu(self.bn(self.conv(x), training=training))


class DepthwiseSeparable(tf.keras.layers.Layer):
    def __init__(self, out_ch, K=3):
        super().__init__()
        self.dw = tf.keras.layers.DepthwiseConv2D(K, padding='same', use_bias=False)
        self.pw = tf.keras.layers.Conv2D(out_ch, 1, use_bias=False)
        self.bn = tf.keras.layers.BatchNormalization()

    def call(self, x, training=False):
        return tf.nn.relu(self.bn(self.pw(self.dw(x)), training=training))


# Parameter count comparison: in_ch=64, out_ch=128, K=3
in_ch, out_ch, K = 64, 128, 3
std_params = in_ch * out_ch * K * K
dws_params = in_ch * K * K + in_ch * out_ch
ratio = std_params / dws_params
print(f'Parameter reduction (in={in_ch}, out={out_ch}, K={K}):')
print(f'  Standard conv:        {std_params:,} params')
print(f'  Depthwise separable:  {dws_params:,} params')
print(f'  Reduction factor:     {ratio:.1f}×')

# Verify output shape: NHWC
x_test = tf.random.normal((1, 32, 32, 16))
out = tf.keras.layers.Conv2D(32, 3, padding='same')(x_test)
print(f'\nConv2D output shape: {out.shape}  (NHWC — height and width preserved with same padding)')

In [ ]:
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.cifar10.load_data()
x_train = x_train_raw.astype('float32') / 255.0
x_test  = x_test_raw.astype('float32')  / 255.0
mean = x_train.mean(axis=(0, 1, 2), keepdims=True)
std  = x_train.std(axis=(0, 1, 2),  keepdims=True) + 1e-7
x_train = (x_train - mean) / std
x_test  = (x_test  - mean) / std
y_train = y_train_raw.squeeze()
y_test  = y_test_raw.squeeze()
print(f'CIFAR-10: {x_train.shape} train, {x_test.shape} test  (NHWC)')

In [ ]:
def build_cnn(use_dws=False):
    block = DepthwiseSeparable if use_dws else ConvBlock
    inp  = tf.keras.Input(shape=(32, 32, 3))
    x    = ConvBlock(32)(inp)           # always standard for first layer
    x    = tf.keras.layers.MaxPool2D()(x)
    x    = block(64)(x)
    x    = tf.keras.layers.MaxPool2D()(x)
    x    = block(128)(x)
    x    = tf.keras.layers.MaxPool2D()(x)
    x    = block(128)(x)
    x    = block(256)(x)
    x    = tf.keras.layers.GlobalAveragePooling2D()(x)
    out  = tf.keras.layers.Dense(10)(x)
    return tf.keras.Model(inp, out)


def train_cnn(model, n_epochs=5, batch_size=128):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'],
    )
    hist = model.fit(
        x_train, y_train,
        validation_data=(x_test, y_test),
        epochs=n_epochs,
        batch_size=batch_size,
        verbose=1,
    )
    return hist


tf.random.set_seed(1)
cnn_std = build_cnn(use_dws=False)
cnn_dws = build_cnn(use_dws=True)
print(f'Standard CNN params: {cnn_std.count_params():,}')
print(f'DWS CNN params:      {cnn_dws.count_params():,}')
print(f'Reduction: {cnn_std.count_params()/cnn_dws.count_params():.1f}×\n')

print('Training standard CNN:')
hist_std = train_cnn(cnn_std)
print('\nTraining DWS CNN:')
hist_dws = train_cnn(cnn_dws)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_std.history['loss'],     'o-', label='Standard CNN')
axes[0].plot(hist_dws.history['loss'],     's-', label='DWS CNN')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist_std.history['val_accuracy'], 'o-', label='Standard CNN')
axes[1].plot(hist_dws.history['val_accuracy'], 's-', label='DWS CNN')
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Section 3 — ResNet with GradientTape

In TensorFlow, gradients are computed explicitly by wrapping the forward pass in a `tf.GradientTape` context and calling `tape.gradient(loss, variables)`. We use this to inspect per-layer gradient norms in plain vs. residual networks.

In [ ]:
class BasicBlock(tf.keras.layers.Layer):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(channels, 3, padding='same', use_bias=False)
        self.bn1   = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(channels, 3, padding='same', use_bias=False)
        self.bn2   = tf.keras.layers.BatchNormalization()

    def call(self, x, training=False):
        out = tf.nn.relu(self.bn1(self.conv1(x), training=training))
        out = self.bn2(self.conv2(out), training=training)
        return tf.nn.relu(out + x)   # residual connection


def build_plain_net(n_blocks=10, channels=32):
    inp  = tf.keras.Input(shape=(32, 32, 3))
    x    = tf.keras.layers.Conv2D(channels, 3, padding='same', use_bias=False, activation='relu')(inp)
    for _ in range(n_blocks):
        x = tf.keras.layers.Conv2D(channels, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)
    out = tf.keras.layers.Dense(10)(x)
    return tf.keras.Model(inp, out)


def build_resnet(n_blocks=10, channels=32):
    inp  = tf.keras.Input(shape=(32, 32, 3))
    x    = tf.keras.layers.Conv2D(channels, 3, padding='same', use_bias=False, activation='relu')(inp)
    for _ in range(n_blocks):
        # Inline residual to keep the Functional API happy
        identity = x
        x = tf.keras.layers.Conv2D(channels, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.Conv2D(channels, 3, padding='same', use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Add()([x, identity])
        x = tf.keras.layers.ReLU()(x)
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)
    out = tf.keras.layers.Dense(10)(x)
    return tf.keras.Model(inp, out)


tf.random.set_seed(2)
plain  = build_plain_net(10)
resnet = build_resnet(10)
print(f'Plain:  {plain.count_params():,} params')
print(f'ResNet: {resnet.count_params():,} params')

In [ ]:
x_batch = tf.constant(x_train[:16])
y_batch = tf.constant(y_train[:16], dtype=tf.int32)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

def get_conv_grad_norms(model):
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss   = loss_fn(y_batch, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    norms = []
    for var, grad in zip(model.trainable_variables, grads):
        if 'conv' in var.name and 'kernel' in var.name and grad is not None:
            norms.append(tf.norm(grad).numpy())
    return norms

norms_plain  = get_conv_grad_norms(plain)
norms_resnet = get_conv_grad_norms(resnet)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, scale in [(axes[0], 'linear'), (axes[1], 'log')]:
    fn = ax.plot if scale == 'linear' else ax.semilogy
    fn(norms_plain,  'o-', label='Plain CNN', linewidth=2, markersize=5)
    fn(norms_resnet, 's-', label='ResNet',    linewidth=2, markersize=5)
    ax.set_xlabel('Conv layer index (shallow → deep)')
    ax.set_ylabel('Gradient norm' + (' (log)' if scale == 'log' else ''))
    ax.set_title(f'Gradient Flow via GradientTape — {scale}')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Plain  — first/last ratio: {norms_plain[0]/max(norms_plain[-1], 1e-30):.2f}')
print(f'ResNet — first/last ratio: {norms_resnet[0]/max(norms_resnet[-1], 1e-30):.2f}')

## Section 4 — Vanilla RNN and LSTM from Scratch

We implement both cells as `tf.keras.layers.Layer` subclasses and wrap them in a manual timestep loop. The key TF difference: gradient inspection uses `GradientTape` rather than `requires_grad`.

In [ ]:
class VanillaRNNCell(tf.keras.layers.Layer):
    def __init__(self, d_h):
        super().__init__()
        self.Wh = tf.keras.layers.Dense(d_h, use_bias=False)
        self.Wx = tf.keras.layers.Dense(d_h)

    def call(self, x_t, h_prev):
        return tf.tanh(self.Wh(h_prev) + self.Wx(x_t))


class LSTMCell(tf.keras.layers.Layer):
    def __init__(self, d_h):
        super().__init__()
        self.Wih = tf.keras.layers.Dense(4 * d_h)
        self.Whh = tf.keras.layers.Dense(4 * d_h, use_bias=False)
        self.d_h = d_h

    def call(self, x_t, h_prev, c_prev):
        gates = self.Wih(x_t) + self.Whh(h_prev)
        i, f, g, o = tf.split(gates, 4, axis=-1)
        i = tf.sigmoid(i); f = tf.sigmoid(f)
        g = tf.tanh(g);    o = tf.sigmoid(o)
        c = f * c_prev + i * g
        h = o * tf.tanh(c)
        return h, c


class RNNModel(tf.keras.Model):
    def __init__(self, d_in, d_h, d_out, cell_type='rnn'):
        super().__init__()
        self.d_h   = d_h
        self.ctype = cell_type
        if cell_type == 'rnn':
            self.cell = VanillaRNNCell(d_h)
        else:
            self.cell = LSTMCell(d_h)
        self.out = tf.keras.layers.Dense(d_out)

    def call(self, x, training=False):   # x: (B, T, d_in)
        B = tf.shape(x)[0]
        h = tf.zeros((B, self.d_h))
        if self.ctype == 'lstm':
            c = tf.zeros((B, self.d_h))
            for t in range(x.shape[1]):
                h, c = self.cell(x[:, t, :], h, c)
        else:
            for t in range(x.shape[1]):
                h = self.cell(x[:, t, :], h)
        return self.out(h)


# Long-sequence synthetic task
tf.random.set_seed(3)
N, SEQ, D = 2000, 200, 8
X_data = tf.random.normal((N, SEQ, D))
y_data = tf.cast(X_data[:, 0, 0] > 0, tf.int32)
split  = 1600
X_tr, y_tr = X_data[:split], y_data[:split]
X_v,  y_v  = X_data[split:], y_data[split:]
print(f'Dataset: {X_tr.shape} train, {X_v.shape} val  (seq_len={SEQ}, signal at t=0)')

In [ ]:
def train_rnn_model(model, n_epochs=10, batch_size=64, lr=1e-3):
    opt     = tf.keras.optimizers.Adam(lr)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    accs    = []
    dataset = tf.data.Dataset.from_tensor_slices((X_tr, y_tr)).shuffle(1600).batch(batch_size)
    for epoch in range(n_epochs):
        for xb, yb in dataset:
            with tf.GradientTape() as tape:
                loss = loss_fn(yb, model(xb, training=True))
            grads = tape.gradient(loss, model.trainable_variables)
            opt.apply_gradients(zip(grads, model.trainable_variables))
        preds = tf.argmax(model(X_v, training=False), axis=1)
        acc   = tf.reduce_mean(tf.cast(tf.equal(tf.cast(preds, tf.int32), y_v), tf.float32)).numpy()
        accs.append(acc)
        print(f'  Epoch {epoch+1}: val_acc={acc:.3f}')
    return accs


tf.random.set_seed(4)
print('Vanilla RNN:')
rnn_model  = RNNModel(D, 64, 2, 'rnn')
rnn_model(X_tr[:1])   # build
accs_rnn   = train_rnn_model(rnn_model)

tf.random.set_seed(4)
print('\nLSTM:')
lstm_model = RNNModel(D, 64, 2, 'lstm')
lstm_model(X_tr[:1])
accs_lstm  = train_rnn_model(lstm_model)

plt.figure(figsize=(8, 4))
plt.plot(accs_rnn,  'o-', label='Vanilla RNN', linewidth=2)
plt.plot(accs_lstm, 's-', label='LSTM',        linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='chance')
plt.xlabel('Epoch'); plt.ylabel('Val accuracy')
plt.title('Long-Sequence Task (TF) — seq_len=200, signal at t=0')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Gradient norms at first timestep using GradientTape
loss_fn2 = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
x_sample = tf.Variable(X_v[:32])
y_sample  = y_v[:32]

for name, model in [('Vanilla RNN', rnn_model), ('LSTM', lstm_model)]:
    with tf.GradientTape() as tape:
        tape.watch(x_sample)
        loss = loss_fn2(y_sample, model(x_sample, training=False))
    grad = tape.gradient(loss, x_sample)   # (B, T, D)
    norm_t0 = tf.norm(grad[:, 0, :]).numpy()
    print(f'{name}: gradient norm at t=0 = {norm_t0:.4e}')

print('\nLSTM gradient should be larger — cell-state path preserves gradients across timesteps.')

## Section 5 — Scaled Dot-Product Attention

Causal masking in TF: use an additive mask of −1e9 for future positions so softmax drives them to ≈0. We verify the saturation behaviour and compare our `MultiHeadAttention` against the built-in `tf.keras.layers.MultiHeadAttention`.

In [ ]:
class ScaledDotProductAttention(tf.keras.layers.Layer):
    def call(self, Q, K, V, mask=None, scale=True):
        """Q, K, V: (..., seq, d_k). Returns (output, weights)."""
        d_k    = tf.cast(tf.shape(K)[-1], tf.float32)
        scores = tf.matmul(Q, K, transpose_b=True)
        if scale:
            scores = scores / tf.math.sqrt(d_k)
        if mask is not None:
            scores = scores + mask * -1e9
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V), weights


sdpa = ScaledDotProductAttention()

# Saturation experiment
d_k_values = [4, 8, 16, 32, 64, 128, 256, 512]
entropies_scaled   = []
entropies_unscaled = []

tf.random.set_seed(5)
for d_k in d_k_values:
    Q = tf.random.normal((64, 16, d_k))
    K = tf.random.normal((64, 16, d_k))
    V = tf.random.normal((64, 16, d_k))
    _, w_s  = sdpa(Q, K, V, scale=True)
    _, w_u  = sdpa(Q, K, V, scale=False)
    eps = 1e-9
    ent_s = -tf.reduce_mean(tf.reduce_sum(w_s * tf.math.log(w_s + eps), axis=-1)).numpy()
    ent_u = -tf.reduce_mean(tf.reduce_sum(w_u * tf.math.log(w_u + eps), axis=-1)).numpy()
    entropies_scaled.append(ent_s)
    entropies_unscaled.append(ent_u)

plt.figure(figsize=(8, 4))
plt.plot(d_k_values, entropies_scaled,   'o-', label='With √d_k scaling',    linewidth=2)
plt.plot(d_k_values, entropies_unscaled, 's-', label='Without √d_k scaling', linewidth=2)
plt.xlabel('d_k'); plt.ylabel('Softmax entropy (nats)')
plt.title('Attention Weight Entropy vs d_k (TF)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f'Entropy at d_k=512 — scaled: {entropies_scaled[-1]:.4f}  unscaled: {entropies_unscaled[-1]:.4f}')

In [ ]:
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads
        self.W_q     = tf.keras.layers.Dense(d_model, use_bias=False)
        self.W_k     = tf.keras.layers.Dense(d_model, use_bias=False)
        self.W_v     = tf.keras.layers.Dense(d_model, use_bias=False)
        self.W_o     = tf.keras.layers.Dense(d_model, use_bias=False)
        self.sdpa    = ScaledDotProductAttention()

    def call(self, x, causal_mask=False):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]
        H, dk = self.n_heads, self.d_k

        def split_heads(t):   # (B, T, D) → (B, H, T, dk)
            t = tf.reshape(t, (B, T, H, dk))
            return tf.transpose(t, [0, 2, 1, 3])

        Q = split_heads(self.W_q(x))
        K = split_heads(self.W_k(x))
        V = split_heads(self.W_v(x))

        mask = None
        if causal_mask:
            # Lower-triangular = 0 (attend), upper = 1 (mask)
            ones  = tf.ones((T, T))
            lower = tf.linalg.band_part(ones, -1, 0)  # keep lower triangle
            mask  = tf.cast(1 - lower, tf.float32)[tf.newaxis, tf.newaxis, :, :]

        out, self._last_weights = self.sdpa(Q, K, V, mask=mask)
        out = tf.transpose(out, [0, 2, 1, 3])         # (B, T, H, dk)
        out = tf.reshape(out, (B, T, H * dk))
        return self.W_o(out)


# Verify shape
tf.random.set_seed(6)
mha   = MultiHeadAttention(d_model=128, n_heads=8)
x_t   = tf.random.normal((4, 10, 128))
out   = mha(x_t)
print(f'MHA output shape: {out.shape}  (expected: (4, 10, 128))')
assert out.shape == (4, 10, 128)

# Compare with tf.keras.layers.MultiHeadAttention
ref = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=16, use_bias=False)
out_ref = ref(x_t, x_t)
print(f'Output std (ours): {tf.math.reduce_std(out).numpy():.4f}  |  ref: {tf.math.reduce_std(out_ref).numpy():.4f}')
print('✓ Same shape and comparable scale')

## Section 6 — Mini Decoder-Only Transformer

We assemble a character-level language model using the custom `MultiHeadAttention` from Section 5. The `training=` flag must be passed explicitly through each sub-layer in a custom `GradientTape` training loop — Keras `model.fit` would do this automatically, but using a manual loop makes the mechanics visible.

In [ ]:
def sinusoidal_pe(T, d_model):
    pe  = np.zeros((T, d_model), dtype=np.float32)
    pos = np.arange(T)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div[:d_model // 2])
    return tf.constant(pe)   # (T, d_model)


# Verify periodicity
pe = sinusoidal_pe(100, 64)
dots_20 = [float(tf.reduce_sum(pe[20] * pe[20 + k])) for k in range(8)]
dots_50 = [float(tf.reduce_sum(pe[50] * pe[50 + k])) for k in range(8)]
print('PE dot products (should match across absolute positions):')
print('  pos=20:', [f'{v:.2f}' for v in dots_20])
print('  pos=50:', [f'{v:.2f}' for v in dots_50])

plt.figure(figsize=(10, 3))
plt.imshow(pe[:50, :64].numpy(), aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar()
plt.title('Sinusoidal Positional Encoding (TF)')
plt.xlabel('Dimension'); plt.ylabel('Position'); plt.tight_layout(); plt.show()

In [ ]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = tf.keras.layers.LayerNormalization(axis=-1)
        self.norm2 = tf.keras.layers.LayerNormalization(axis=-1)
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.ffn   = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation='gelu'),
            tf.keras.layers.Dense(d_model),
        ])
        self.drop  = tf.keras.layers.Dropout(dropout)

    def call(self, x, causal_mask=True, training=False):
        x = x + self.drop(self.attn(self.norm1(x), causal_mask=causal_mask), training=training)
        x = x + self.drop(self.ffn(self.norm2(x),  training=training), training=training)
        return x


class MiniGPT(tf.keras.Model):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=3,
                 d_ff=256, max_len=128, dropout=0.1):
        super().__init__()
        self.embed  = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pe     = sinusoidal_pe(max_len, d_model)
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff, dropout)
                       for _ in range(n_layers)]
        self.norm   = tf.keras.layers.LayerNormalization(axis=-1)
        self.head   = tf.keras.layers.Dense(vocab_size)
        self.drop   = tf.keras.layers.Dropout(dropout)

    def call(self, idx, training=False):   # idx: (B, T)
        T = tf.shape(idx)[1]
        x = self.drop(self.embed(idx) + self.pe[:T], training=training)
        for block in self.blocks:
            x = block(x, causal_mask=True, training=training)
        return self.head(self.norm(x))


# Character-level dataset
TEXT = (
    "To be, or not to be, that is the question: whether 'tis nobler in the mind to suffer "
    "the slings and arrows of outrageous fortune, or to take arms against a sea of troubles. "
    "All the world's a stage, and all the men and women merely players. "
    "Friends, Romans, countrymen, lend me your ears. I come to bury Caesar, not to praise him. "
    "What light through yonder window breaks? It is the east, and Juliet is the sun. "
) * 10

chars    = sorted(set(TEXT))
stoi     = {ch: i for i, ch in enumerate(chars)}
vocab_sz = len(chars)
data_np  = np.array([stoi[c] for c in TEXT], dtype=np.int32)
CTX      = 64
print(f'Vocab size: {vocab_sz}  |  Dataset: {len(data_np)} chars  |  Context: {CTX}')

In [ ]:
def get_batch_tf(data, batch_size=32, ctx=CTX):
    idx = np.random.randint(0, len(data) - ctx, size=batch_size)
    x   = np.stack([data[i:i+ctx]   for i in idx])
    y   = np.stack([data[i+1:i+ctx+1] for i in idx])
    return tf.constant(x), tf.constant(y)


tf.random.set_seed(7)
np.random.seed(7)
gpt_model = MiniGPT(vocab_sz, d_model=128, n_heads=4, n_layers=3, d_ff=256, max_len=CTX)
# Build the model
xb_init, _ = get_batch_tf(data_np)
_ = gpt_model(xb_init, training=False)
print(f'MiniGPT params: {gpt_model.count_params():,}')

opt         = tf.keras.optimizers.AdamW(learning_rate=3e-3, weight_decay=1e-4)
loss_fn_lm  = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
losses_gpt  = []
N_STEPS     = 400

for step in range(N_STEPS):
    xb, yb = get_batch_tf(data_np)
    with tf.GradientTape() as tape:
        logits = gpt_model(xb, training=True)   # (B, T, vocab_sz)
        # Flatten for loss: (B*T, vocab_sz) vs (B*T,)
        loss   = loss_fn_lm(
            tf.reshape(yb, (-1,)),
            tf.reshape(logits, (-1, vocab_sz))
        )
    grads = tape.gradient(loss, gpt_model.trainable_variables)
    opt.apply_gradients(zip(grads, gpt_model.trainable_variables))
    losses_gpt.append(loss.numpy())
    if (step + 1) % 100 == 0:
        print(f'  Step {step+1}/{N_STEPS}: loss={loss.numpy():.4f}')

plt.figure(figsize=(8, 3))
w = 20
smoothed = np.convolve(losses_gpt, np.ones(w)/w, mode='valid')
plt.plot(losses_gpt, alpha=0.3, color='steelblue')
plt.plot(range(w-1, len(losses_gpt)), smoothed, color='steelblue', linewidth=2, label='smoothed')
plt.xlabel('Step'); plt.ylabel('Cross-entropy loss')
plt.title('Mini Transformer Training Loss (TF)'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Per-head attention heatmaps
x_sample = tf.constant(data_np[:CTX][np.newaxis, :])   # (1, CTX)
_ = gpt_model(x_sample, training=False)
attn_weights = gpt_model.blocks[-1].attn._last_weights  # (1, n_heads, T, T)

n_heads = attn_weights.shape[1]
fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
sample_text = TEXT[:CTX]
tick_pos    = list(range(0, CTX, 8))
tick_labels = list(sample_text[::8])

for h, ax in enumerate(axes):
    w = attn_weights[0, h].numpy()
    im = ax.imshow(w, cmap='viridis', aspect='auto', vmin=0)
    ax.set_title(f'Head {h+1}', fontsize=11)
    ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Per-Head Attention Patterns — final block (TF)', y=1.02)
plt.tight_layout(); plt.show()
print('Compare head patterns against PyTorch Lab to verify cross-framework parity.')